<a href="https://colab.research.google.com/github/MayerT1/PIPECAST/blob/flood_mapping/flood_mapping/notebooks/PIPECAST_hydrafloods_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1: Housekeeping and obtaininng ICEYE image

In [ ]:
my_gdrive_folder = 'drive/MyDrive/watchduty/'
#my_gee_folder = 'users/mickymags/watchduty/'

In [ ]:
!pip install ipyleaflet==0.18.2 geemap hydrafloods     # Install hydrafloods and its relevant dependencies
!pip install geemap

In [ ]:
from hydrafloods import corrections
import hydrafloods as hf
import ee
import geemap
import google.colab
from osgeo import gdal
import numpy as np

In [ ]:
ee.Authenticate()
ee.Initialize(project = 'servir-sco-assets')

In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive/')

In [ ]:
import os
import shutil

In [ ]:
ls

In [ ]:
os.chdir(my_gdrive_folder)

In [ ]:
ls

In [ ]:
in_file = 'louisiana_20200702.tif'
out_file = 'EDIT_louisiana_20200702.tif'

In [ ]:
ls

In [ ]:
cd ICEYE_DATA

In [ ]:
ls

In [ ]:
shutil.copy('louisiana_20200702.tif', 'edited_louisiana_20200702.tif')

In [ ]:
in_file = 'edited_louisiana_20200702.tif'
out_file = 'EDIT2_louisiana_20200702.tif'

In [ ]:
ls

In [ ]:
!gdal_edit.py -a_srs EPSG:4326 edited_louisiana_20200702.tif

In [ ]:
!gdal_edit.py -a_ullr -91.65 30.175 -90.986 30.853 edited_louisiana_20200702.tif

In [ ]:
gdal.Warp('ULT_louisiana_20200702.tif', 'edited_louisiana_20200702.tif', dstSRS='EPSG:4326')

In [ ]:
ls

In [ ]:
pwd

In [ ]:
info = gdal.Info('ULT_louisiana_20200702.tif')

In [ ]:
inc_angle = info[info.find('INCIDENCE_CENTER'):info.find('INCIDENCE_CENTER')+30].split('=')[1]

In [ ]:
new_ds = gdal.Open('ULT_louisiana_20200702.tif', gdal.GA_ReadOnly)

In [ ]:
geotransform = new_ds.GetGeoTransform()
proj = new_ds.GetProjection()
xsize = new_ds.RasterXSize
ysize = new_ds.RasterYSize
nbands = new_ds.RasterCount

# Create output dataset with one extra band
driver = gdal.GetDriverByName("GTiff")
out_ds = driver.Create('louisiana_20200702_1525.tif', xsize, ysize, nbands + 1, gdal.GDT_Float32)

# Copy georeferencing
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(proj)

# Copy Original Band
in_band = new_ds.GetRasterBand(1)
data = in_band.ReadAsArray()
out_band = out_ds.GetRasterBand(1)
out_band.WriteArray(data)
out_band.SetDescription("VV")

# Preserve NoData and description if available
nodata = in_band.GetNoDataValue()
if nodata is not None:
  out_band.SetNoDataValue(nodata)

desc = in_band.GetDescription()
if desc:
  out_band.SetDescription(desc)

# Create constant incidence angle band
const_array = np.full((ysize, xsize), inc_angle, dtype = np.float32)
angle_band = out_ds.GetRasterBand(2)
angle_band.WriteArray(const_array)
angle_band.SetDescription("angle")

# Save and close
out_ds.FlushCache()
out_ds = None
src_ds = None

In [ ]:
ls l*

# Upload dataset to GEE

In [ ]:
xsize

# Arkansas Image

In [ ]:
pwd

In [ ]:
ls

In [ ]:
cd ICEYE_DATA

In [ ]:
shutil.copy('arkansas_20200331.tif', 'arkansas_edit_0918.tif')

In [ ]:
ls ark*

In [ ]:
shutil.copy('arkansas_20200331.tif', 'arkansas_take3.tif')

In [ ]:
ls ark*

In [ ]:
!gdal_edit.py -a_srs EPSG:4326 arkansas_take3.tif

In [ ]:
!gdal_edit.py -a_ullr -89.767 35.481 -90.47 34.805 arkansas_take3.tif

In [ ]:
ls arkansas_ta*

In [ ]:
shutil.copy('arkansas_take3.tif', 'arkansas_take3_final.tif')

In [ ]:
ls arkansas_take3*

In [ ]:
import time

In [ ]:
time.sleep(60)

In [ ]:
shutil.copy('arkansas_20200331.tif', 'arkansas_trying.tif')

In [ ]:
!gdal_edit.py -a_srs EPSG:4326 arkansas_trying.tif

In [ ]:
!gdal_edit.py -a_ullr -89.767 35.481 -90.47 34.805 arkansas_trying.tif

In [ ]:
arkansas_info = gdal.Info('arkansas_take3.tif')

In [ ]:
ark_inc_angle = arkansas_info[arkansas_info.find('INCIDENCE_CENTER'):arkansas_info.find('INCIDENCE_CENTER')+30].split('=')[1]
ark_inc_angle

In [ ]:
art3_ds = gdal.Open('arkansas_take3.tif', gdal.GA_ReadOnly)

In [ ]:
art3_ds

In [ ]:
geotransform = art3_ds.GetGeoTransform()
proj = art3_ds.GetProjection()
xsize = art3_ds.RasterXSize
ysize = art3_ds.RasterYSize
nbands = art3_ds.RasterCount

# Create output dataset with one extra band
driver = gdal.GetDriverByName("GTiff")
out_ds = driver.Create('arkansas_take3.tif', xsize, ysize, nbands + 1, gdal.GDT_Float32)

# Copy georeferencing
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(proj)

# Copy Original Band
in_band = art3_ds.GetRasterBand(1)
data = in_band.ReadAsArray()
out_band = out_ds.GetRasterBand(1)
out_band.WriteArray(data)
out_band.SetDescription("VV")

# Preserve Nodata and description if available
nodata = in_band.GetNoDataValue()
if nodata is not None:
  out_band.SetNoDataValue(nodata)

desc = in_band.GetDescription()
if desc:
  out_band.SetDescription(desc)

# Create constant incidence angle band
const_array = np.full((ysize, xsize), ark_inc_angle, dtype = np.float32)
angle_band = out_ds.GetRasterBand(2)
angle_band.WriteArray(const_array)
angle_band.SetDescription("angle")

# Save and close
out_ds.FlushCache()
out_ds = None
src_ds = None

In [ ]:
shutil.copy('arkansas_take3.tif', 'arkansas_take3_update.tif')

In [ ]:
!gdal_edit.py -a_ullr -90.47 35.481 -89.767 34.805 arkansas_edit_0918.tif

In [ ]:
!gdal_edit.py -a_srs EPSG:4326 arkansas_sandbox.tif

In [ ]:
ls ark*

In [ ]:
in_file = 'edited_arkansas_20200331.tif'
out_file = 'inter_arkansas_20200331.tif'

In [ ]:
!gdal_edit.py -a_srs EPSG:4326 arkansas_edit_0918.tif

In [ ]:
!gdal_edit.py -a_ullr -90.47 35.481 -89.767 34.805 arkansas_edit_0918.tif

In [ ]:
gdal.Warp('arkansas_FiNaL_0918.tif', 'arkansas_edit_0918.tif', dstSRS='EPSG:4326')

In [ ]:
ls ark*

In [ ]:
arkansas_info = gdal.Info('arkansas_FiNaL_0918.tif')

In [ ]:
inc_angle = arkansas_info[arkansas_info.find('INCIDENCE_CENTER'):arkansas_info.find('INCIDENCE_CENTER')+30].split('=')[1]

In [ ]:
inc_angle

In [ ]:
ar_ds = gdal.Open('arkansas_FiNaL_0918.tif', gdal.GA_ReadOnly)

In [ ]:
ar_ds = gdal.Open('arkansas_FiNaL_0918.tif', gdal.GA_ReadOnly)

In [ ]:
ls ark*

In [ ]:
gdal.Warp('semifinal_arkansas_0918.tif', 'arkansas_FiNaL_0918.tif', dstSRS='EPSG:4326')

# Alabama Image

In [ ]:
ls

In [ ]:
shutil.copy('alabama_20230721.tif', 'edited_alabama_20230721.tif')

In [ ]:
in_file = 'edited_alabama_20230721.tif'
out_file = 'EDIT2_alabama_20230721.tif'

In [ ]:
ls

In [ ]:
!gdal_edit.py -a_srs EPSG:4326 edited_alabama_20230721.tif

In [ ]:
!gdal_edit.py -a_ullr -86.976 34.367 -86.485 35.174 edited_alabama_20230721.tif

In [ ]:
gdal.Warp('ULT_alabama_20230721.tif', 'edited_alabama_20230721.tif', dstSRS='EPSG:4326')

In [ ]:
inc_angle = info[info.find('INCIDENCE_CENTER'):info.find('INCIDENCE_CENTER')+30].split('=')[1]

In [ ]:
al_ds = gdal.Open('ULT_alabama_20230721.tif', gdal.GA_ReadOnly)

In [ ]:
geotransform = al_ds.GetGeoTransform()
proj = al_ds.GetProjection()
xsize = al_ds.RasterXSize
ysize = al_ds.RasterYSize
nbands = al_ds.RasterCount

# Create output dataset with one extra band
driver = gdal.GetDriverByName("GTiff")
out_ds = driver.Create('alabama_20230721_1649.tif', xsize, ysize, nbands + 1, gdal.GDT_Float32)

# Copy georeferencing
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(proj)

# Copy Original Band
in_band = al_ds.GetRasterBand(1)
data = in_band.ReadAsArray()
out_band = out_ds.GetRasterBand(1)
out_band.WriteArray(data)
out_band.SetDescription("VV")

# Preserve Nodata and description if available
nodata = in_band.GetNoDataValue()
if nodata is not None:
  out_band.SetNoDataValue(nodata)

desc = in_band.GetDescription()
if desc:
  out_band.SetDescription(desc)

# Create constant incidence angle band
const_array = np.full((ysize, xsize), inc_angle, dtype = np.float32)
angle_band = out_ds.GetRasterBand(2)
angle_band.WriteArray(const_array)
angle_band.SetDescription("angle")

# Save and close
out_ds.FlushCache()
out_ds = None
src_ds = None